# Linear Regression on Census Income Data via BigQuery
**Task ID:** `linreg_lvl5_bq_housing`  
**Series:** Linear Regression, Level 5  
**Protocol:** `pytorch_task_v1`

## Overview
Load US Census Adult Income data **from Google BigQuery**, train multivariate
linear regression with L2 regularization in PyTorch to predict `education_num`
from other numeric features, and compare to an sklearn `LinearRegression` baseline.

## Math
- $h(X) = X W^T + b$
- MSE: $L = \frac{1}{N}\sum_i(\hat{y}_i - y_i)^2$
- L2 regularization via `weight_decay` in SGD: effectively adds $\lambda \|W\|^2$ to loss

In [ ]:
import sys
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from google.cloud import bigquery

print(f"PyTorch version: {torch.__version__}")

## Required Functions (`pytorch_task_v1` protocol)

In [ ]:
def get_task_metadata():
    return {
        "task_id": "linreg_lvl5_bq_housing",
        "algorithm": "Linear Regression (BigQuery Census + Feature Engineering)",
        "series": "Linear Regression",
        "level": 5,
        "interface_protocol": "pytorch_task_v1",
    }

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

print(f"Device: {get_device()}")
print(f"Task: {get_task_metadata()['task_id']}")

## Load Data from BigQuery

**Table:** `bigquery-public-data.ml_datasets.census_adult_income`

We query numeric features directly from BigQuery using SQL, then convert
to PyTorch tensors for training.

In [ ]:
GCP_PROJECT_ID = "gen-lang-client-0916541599"

def load_data_from_bigquery(project_id=GCP_PROJECT_ID):
    """Query Census Adult Income data from BigQuery public dataset."""
    client = bigquery.Client(project=project_id)
    query = """
    SELECT
        age,
        education_num,
        hours_per_week,
        capital_gain,
        capital_loss
    FROM `bigquery-public-data.ml_datasets.census_adult_income`
    WHERE education_num IS NOT NULL
    """
    df = client.query(query).to_dataframe()
    print(f"Loaded {len(df)} rows from BigQuery table: census_adult_income")
    print(f"Columns: {list(df.columns)}")
    print(f"Target: education_num (range {df['education_num'].min()}-{df['education_num'].max()})")
    return df

# --- ARCHIVED: sklearn fallback (same UCI dataset, no BigQuery needed) ---
# def load_data_fallback():
#     from sklearn.datasets import fetch_openml
#     adult = fetch_openml('adult', version=2, as_frame=True)
#     df = adult.frame[['age', 'education-num', 'hours-per-week',
#                        'capital-gain', 'capital-loss']].copy()
#     df.columns = ['age', 'education_num', 'hours_per_week',
#                   'capital_gain', 'capital_loss']
#     return df.dropna()

In [ ]:
def make_dataloaders(cfg):
    df = load_data_from_bigquery()

    # Target: education_num; Features: age, hours_per_week, capital_gain, capital_loss
    feature_cols = ['age', 'hours_per_week', 'capital_gain', 'capital_loss']
    target_col = 'education_num'

    X_np = df[feature_cols].values.astype(np.float32)
    y_np = df[target_col].values.astype(np.float32)

    # Train/val split
    n = len(X_np)
    n_val = int(n * cfg.get("val_ratio", 0.2))
    idx = np.random.permutation(n)
    train_idx, val_idx = idx[n_val:], idx[:n_val]

    # Standardize features using TRAIN stats only
    X_mean = X_np[train_idx].mean(axis=0)
    X_std = X_np[train_idx].std(axis=0) + 1e-8
    X_np = (X_np - X_mean) / X_std

    X = torch.from_numpy(X_np)
    y = torch.from_numpy(y_np).unsqueeze(1)

    train_ds = TensorDataset(X[train_idx], y[train_idx])
    val_ds = TensorDataset(X[val_idx], y[val_idx])

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Features: {len(feature_cols)}")

    return (
        DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True),
        DataLoader(val_ds, batch_size=cfg["batch_size"]),
        {"X_mean": X_mean, "X_std": X_std, "feature_cols": feature_cols,
         "X_train": X_np[train_idx], "y_train": y_np[train_idx],
         "X_val": X_np[val_idx], "y_val": y_np[val_idx]},
    )

In [ ]:
def build_model(cfg):
    return nn.Linear(cfg["n_features"], 1)

def train(model, train_loader, val_loader, cfg, device):
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=cfg["lr"],
                          momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    epochs = cfg["epochs"]
    loss_history = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X_b)
        loss_history.append(epoch_loss / len(train_loader.dataset))

        if (epoch + 1) % 50 == 0:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X_b, y_b in val_loader:
                    X_b, y_b = X_b.to(device), y_b.to(device)
                    val_loss += criterion(model(X_b), y_b).item() * len(X_b)
            val_loss /= len(val_loader.dataset)
            print(f"Epoch {epoch+1}/{epochs} | Train MSE: {loss_history[-1]:.4f} | Val MSE: {val_loss:.4f}")

    return loss_history

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_b, y_b in dataloader:
            X_b = X_b.to(device)
            all_preds.append(model(X_b).cpu())
            all_targets.append(y_b)
    preds = torch.cat(all_preds).squeeze()
    targets = torch.cat(all_targets).squeeze()
    mse = torch.mean((preds - targets) ** 2).item()
    ss_res = torch.sum((targets - preds) ** 2).item()
    ss_tot = torch.sum((targets - targets.mean()) ** 2).item()
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {"mse": mse, "rmse": math.sqrt(mse), "r2": r2}

def predict(model, X, device):
    model.eval()
    with torch.no_grad():
        return model(X.to(device)).cpu()

def save_artifacts(outputs, cfg):
    out_dir = cfg.get("output_dir", "./output")
    os.makedirs(out_dir, exist_ok=True)
    saveable = {k: v for k, v in outputs.items() if isinstance(v, (int, float, str, dict, list))}
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(saveable, f, indent=2, default=str)
    print(f"Metrics saved to {out_dir}")

## Configuration

In [ ]:
cfg = {
    "n_features": 4,
    "batch_size": 128,
    "val_ratio": 0.2,
    "epochs": 200,
    "lr": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.01,  # L2 regularization
    "r2_threshold": 0.01,  # Weak relationship — education_num has low linear correlation with these features
    "output_dir": "./output/linreg_lvl5_bq_housing",
}

## Train

In [ ]:
set_seed(42)
device = get_device()

train_loader, val_loader, data_info = make_dataloaders(cfg)
model = build_model(cfg).to(device)

print("\n--- Training PyTorch Model (SGD + L2 Regularization) ---")
loss_history = train(model, train_loader, val_loader, cfg, device)

## Evaluate & sklearn Comparison

In [ ]:
train_metrics = evaluate(model, train_loader, device)
val_metrics = evaluate(model, val_loader, device)

print(f"PyTorch | Train MSE: {train_metrics['mse']:.4f}  R2: {train_metrics['r2']:.4f}")
print(f"PyTorch | Val   MSE: {val_metrics['mse']:.4f}  R2: {val_metrics['r2']:.4f}")

# sklearn comparison — does our PyTorch model match a closed-form solution?
from sklearn.linear_model import LinearRegression
sklearn_model = LinearRegression()
sklearn_model.fit(data_info["X_train"], data_info["y_train"])
sklearn_preds = sklearn_model.predict(data_info["X_val"])
sklearn_mse = float(np.mean((sklearn_preds - data_info["y_val"]) ** 2))
ss_res_sk = float(np.sum((data_info["y_val"] - sklearn_preds) ** 2))
ss_tot_sk = float(np.sum((data_info["y_val"] - data_info["y_val"].mean()) ** 2))
sklearn_r2 = 1.0 - ss_res_sk / ss_tot_sk

print(f"\nsklearn | Val   MSE: {sklearn_mse:.4f}  R2: {sklearn_r2:.4f}")

r2_diff = abs(val_metrics["r2"] - sklearn_r2)
print(f"\nR2 difference (PyTorch vs sklearn): {r2_diff:.4f}")
print("Note: Low R2 is expected — education_num has weak linear dependence on age/hours/capital.")

## Save Artifacts & Quality Assertions

In [ ]:
outputs = {
    "loss_history": loss_history,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics,
    "sklearn_val_mse": sklearn_mse,
    "sklearn_val_r2": sklearn_r2,
}
save_artifacts(outputs, cfg)

print("\n" + "=" * 60)
print("Quality Assertions...")
print("=" * 60)

assert val_metrics["r2"] > cfg["r2_threshold"], \
    f"Val R2 {val_metrics['r2']:.4f} < {cfg['r2_threshold']}"
print(f"PASS: Val R2 {val_metrics['r2']:.4f} > {cfg['r2_threshold']}")

assert r2_diff < 0.10, \
    f"PyTorch vs sklearn R2 difference {r2_diff:.4f} >= 0.10"
print(f"PASS: PyTorch-sklearn R2 gap {r2_diff:.4f} < 0.10 (models agree)")

print("\n=== ALL CHECKS PASSED ===")
exit_code = 0
print(f"sys.exit({exit_code})")